In [6]:
import os
import glob
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import random
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
import pprint
import pyspark
import pyspark.sql.functions as F
import argparse

from pyspark.sql.functions import col
from pyspark.sql.types import StringType, IntegerType, FloatType, DateType

In [ ]:
def process_bronze_table(snapshot_date_str, bronze_clickstream_directory, bronze_attributes_directory, bronze_financials_directory, spark):
    snapshot_date = datetime.strptime(snapshot_date_str, "%Y-%m-%d")
    # snapshot_date_formatted = snapshot_date.strftime("%d/%m/%Y")

    # feature_clickstream
    clickstream_df = spark.read.csv("../data/feature_clickstream.csv", sep=";", header=True, inferSchema=True).filter(col('snapshot_date') == snapshot_date)
    print(snapshot_date_str + 'row count:', clickstream_df.count())
    
    partition_name = "bronze_clickstream_" + snapshot_date_str.replace('-','_') + '.csv'
    filepath = bronze_clickstream_directory + partition_name
    clickstream_df.toPandas().to_csv(filepath, index=False)
    print('saved to:', filepath)

    # features_attributes
    attributes_df = spark.read.csv("../data/features_attributes.csv", sep=";", header=True, inferSchema=True).filter(col('snapshot_date') == snapshot_date)
    print(snapshot_date_str + 'row count:', attributes_df.count())
    
    partition_name = "bronze_attributes_" + snapshot_date_str.replace('-','_') + '.csv'
    filepath = bronze_attributes_directory + partition_name
    attributes_df.toPandas().to_csv(filepath, index=False)
    print('saved to:', filepath)

    # features_financials
    financials_df = spark.read.csv("../data/features_financials.csv", sep=";", header=True, inferSchema=True).filter(col('snapshot_date') == snapshot_date)
    print(snapshot_date_str + 'row count:', financials_df.count())
    
    partition_name = "bronze_financials_" + snapshot_date_str.replace('-','_') + '.csv'
    filepath = bronze_financials_directory + partition_name
    financials_df.toPandas().to_csv(filepath, index=False)
    print('saved to:', filepath)

    return clickstream_df, attributes_df, financials_df